In [2]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.datasets import fetch_california_housing
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.preprocessing import StandardScaler


In [3]:

# -------------------------------
# 1. Load Data
# -------------------------------
housing = fetch_california_housing()
X = pd.DataFrame(housing.data, columns=housing.feature_names)
y = housing.target


In [4]:

# -------------------------------
# 2. Train-Test Split
# -------------------------------
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)


In [5]:

# -------------------------------
# 3. Scaling
# -------------------------------
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)


In [6]:

# -------------------------------
# 4. Models
# -------------------------------
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor

# Linear Regression
lin_reg = LinearRegression()
lin_reg.fit(X_train_scaled, y_train)

# Random Forest (Ensemble)
rf_reg = RandomForestRegressor(n_estimators=100, random_state=42)
rf_reg.fit(X_train_scaled, y_train)

# Gradient Boosting
gb_reg = GradientBoostingRegressor()
gb_reg.fit(X_train_scaled, y_train)

# XGBoost (install if needed: pip install xgboost)
from xgboost import XGBRegressor
xgb_reg = XGBRegressor()
xgb_reg.fit(X_train_scaled, y_train)


,objective,'reg:squarederror'
,base_score,None
,booster,None
,callbacks,None
,colsample_bylevel,None
,colsample_bynode,None
,colsample_bytree,None
,device,None
,early_stopping_rounds,None
,enable_categorical,True
,eval_metric,None


In [7]:

# -------------------------------
# 5. Predictions
# -------------------------------
y_pred_lr = lin_reg.predict(X_test_scaled)
y_pred_rf = rf_reg.predict(X_test_scaled)
y_pred_gb = gb_reg.predict(X_test_scaled)
y_pred_xgb = xgb_reg.predict(X_test_scaled)


In [8]:

# -------------------------------
# 6. Evaluation Function
# -------------------------------
def evaluate(name, y_test, y_pred):
    mse = mean_squared_error(y_test, y_pred)
    mae = mean_absolute_error(y_test, y_pred)
    r2 = r2_score(y_test, y_pred)
    print(f"{name} -> MSE={mse:.4f}, MAE={mae:.4f}, R2={r2:.4f}")


In [9]:

# -------------------------------
# 7. Evaluate All Models
# -------------------------------
evaluate("Linear Regression", y_test, y_pred_lr)
evaluate("Random Forest", y_test, y_pred_rf)
evaluate("Gradient Boosting", y_test, y_pred_gb)
evaluate("XGBoost", y_test, y_pred_xgb)


Linear Regression -> MSE=0.5559, MAE=0.5332, R2=0.5758
Random Forest -> MSE=0.2552, MAE=0.3274, R2=0.8053
Gradient Boosting -> MSE=0.2940, MAE=0.3717, R2=0.7756
XGBoost -> MSE=0.2140, MAE=0.3107, R2=0.8367


In [10]:

# -------------------------------
# 8. Hyperparameter Tuning (Example)
# -------------------------------
print("\n--- Hyperparameter Tuning (Random Forest) ---")

rf_50 = RandomForestRegressor(n_estimators=50, random_state=42)
rf_200 = RandomForestRegressor(n_estimators=200, random_state=42)

rf_50.fit(X_train_scaled, y_train)
rf_200.fit(X_train_scaled, y_train)

print("R2 with 50 trees:", r2_score(y_test, rf_50.predict(X_test_scaled)))
print("R2 with 200 trees:", r2_score(y_test, rf_200.predict(X_test_scaled)))



--- Hyperparameter Tuning (Random Forest) ---
R2 with 50 trees: 0.8038755994657409
R2 with 200 trees: 0.8063074586513359


In [11]:

# -------------------------------
# 9. Cross Validation
# -------------------------------
print("\n--- Cross Validation (Random Forest) ---")

cv_scores = cross_val_score(rf_reg, X, y, cv=5, scoring='r2')
print("Cross-validation scores:", cv_scores)
print("Average R2:", np.mean(cv_scores))



--- Cross Validation (Random Forest) ---
Cross-validation scores: [0.51688816 0.70429935 0.74214734 0.63501894 0.68223973]
Average R2: 0.6561187027256853


In [12]:

# -------------------------------
# 10. Model Selection
# -------------------------------
print("\n--- Model Selection ---")

models = {
    "Linear Regression": lin_reg,
    "Random Forest": rf_reg,
    "Gradient Boosting": gb_reg,
    "XGBoost": xgb_reg
}

results = {}

for name, model in models.items():
    pred = model.predict(X_test_scaled)
    score = r2_score(y_test, pred)
    results[name] = score
    print(f"{name}: R2 = {score:.4f}")

best_model = max(results, key=results.get)
print(f"\n✅ Best Model: {best_model} with R2 = {results[best_model]:.4f}")



--- Model Selection ---
Linear Regression: R2 = 0.5758
Random Forest: R2 = 0.8053
Gradient Boosting: R2 = 0.7756
XGBoost: R2 = 0.8367

✅ Best Model: XGBoost with R2 = 0.8367
